In [2]:
import pandas as pd

In [4]:
df = pd.read_csv('../data/results_gpt_4omini.csv')
df.head()

,prompt,result,references,tokens
0,"I want to write an article about: ""Common fair...",Several studies demonstrate the mathematical i...,['https://jmlr.org/beta/papers/v24/22-1511.htm...,NaN
1,"I want to write an article about: ""Machine Lea...",Machine learning models can exhibit bias even ...,['https://www.forbes.com/sites/aparnadhinakara...,NaN
2,"I want to write an article about: ""Evaluation ...",Evaluating and mitigating fairness solely thro...,['https://arxiv.org/abs/2006.09663?utm_source=...,NaN
3,"I want to write an article about: ""Benchmark c...",Several studies highlight how US-centric bench...,['https://davidnowak.me/why-your-ai-benchmarks...,NaN
4,"I want to write an article about: ""Word embedd...",Several studies have demonstrated that word em...,['https://arxiv.org/abs/1903.03862?utm_source=...,NaN


In [5]:
df['result'].iloc[56]

"The rapid evolution of AI prompting tactics necessitates continuous red-teaming to effectively identify and mitigate emerging vulnerabilities. Traditional static policies often fall short in addressing these dynamic threats. For instance, a prominent hacker group has called for a major overhaul of AI security practices, arguing that current systems remain too vulnerable. ([axios.com](https://www.axios.com/2025/02/11/ai-security-revamp-def-con?utm_source=openai)) Similarly, a survey by Accenture revealed that 36% of companies admit AI is advancing faster than their security capabilities, highlighting the need for adaptive security measures. ([axios.com](https://www.axios.com/2025/06/26/accenture-executives-cybersecurity-ai-plans?utm_source=openai))\n\nTo address these challenges, organizations are increasingly adopting adaptive red-teaming strategies. CrowdStrike, for example, launched AI Red Team Services to proactively identify and mitigate vulnerabilities in AI systems, emphasizing 

In [7]:
import re
from urllib.parse import urlsplit, urlunsplit
import pandas as pd

# Regex general para URLs (http/https), evita capturar paréntesis/llaves/corchetes al final
_URL_RE = re.compile(r'https?://[^\s<>"\']+')

def _clean_url(u: str) -> str:
    """
    Limpia basura común al final: ), ], }, ., ,, ;, :
    y normaliza levemente.
    """
    u = u.strip()

    # Recorta cierres típicos que se pegan en Markdown o puntuación final
    while u and u[-1] in ')]}.,;:':
        u = u[:-1]

    # Opcional: normaliza (sin tocar query/utm)
    # (esto evita cosas raras como espacios u otros, pero es suave)
    parts = urlsplit(u)
    return urlunsplit(parts)

def extract_urls_from_text(text) -> list[str]:
    """
    Devuelve una lista (orden de aparición) con todas las URLs encontradas en el texto.
    Maneja NaN/None.
    """
    if text is None:
        return []
    # Pandas puede traer NaN (float)
    if isinstance(text, float) and pd.isna(text):
        return []

    s = str(text)

    urls = []
    for m in _URL_RE.finditer(s):
        urls.append(_clean_url(m.group(0)))

    # De-duplicar manteniendo orden
    seen = set()
    out = []
    for u in urls:
        if u and u not in seen:
            seen.add(u)
            out.append(u)
    return out

# --- Ejemplo de uso con DataFrame ---
# df["urls"] tendrá una lista por fila con todas las URLs encontradas en df["texto"]
# df["urls"] = df["texto"].apply(extract_urls_from_text)

# Si quieres una sola lista con todas las URLs del dataframe (aplanada):
# all_urls = [u for lst in df["urls"] for u in lst]

In [8]:
df["urls_clean"] = df["result"].apply(extract_urls_from_text)

In [10]:
df['urls_clean'].iloc[56]

['https://www.axios.com/2025/02/11/ai-security-revamp-def-con?utm_source=openai',
 'https://www.axios.com/2025/06/26/accenture-executives-cybersecurity-ai-plans?utm_source=openai',
 'https://www.crowdstrike.com/en-us/press-releases/crowdstrike-launches-ai-red-team-services-secure-ai-systems/?utm_source=openai',
 'https://learn.microsoft.com/en-us/security/ai-red-team/training?utm_source=openai',
 'https://arxiv.org/abs/2510.02677?utm_source=openai',
 'https://www.itpro.com/technology/artificial-intelligence/openai-turns-to-red-teamers-to-prevent-malicious-chatgpt-use-as-company-warns-future-models-could-pose-high-security-risk?utm_source=openai']

In [13]:
import re
import pandas as pd
import requests
from functools import lru_cache

DOI_RE = re.compile(r'10\.\d{4,9}/[^\s"<>]+', re.I)

def extract_doi(text_or_url: str) -> str | None:
    if not text_or_url:
        return None
    m = DOI_RE.search(str(text_or_url))
    return m.group(0).rstrip(').,;:]') if m else None

@lru_cache(maxsize=50_000)
def crossref_metadata(doi: str) -> dict | None:
    r = requests.get(f"https://api.crossref.org/works/{doi}", timeout=20, headers={"User-Agent": "link-classifier/1.0"})
    if r.status_code != 200:
        return None
    return r.json().get("message")

def classify_link_peer_review(url: str) -> dict:
    u = (url or "").lower()

    # Preprints: normalmente no peer-reviewed
    if any(d in u for d in ["arxiv.org", "ssrn.com", "biorxiv.org", "medrxiv.org"]):
        return {"label": "not_refereed", "reason": "preprint_server", "confidence": 0.95}

    doi = extract_doi(url)
    if not doi:
        return {"label": "unknown", "reason": "no_doi", "confidence": 0.3}

    meta = crossref_metadata(doi)
    if not meta:
        return {"label": "unknown", "reason": "doi_not_resolved", "confidence": 0.3}

    work_type = meta.get("type")  # e.g. journal-article, proceedings-article
    container = (meta.get("container-title") or [None])[0]
    publisher = meta.get("publisher")
    issn = meta.get("ISSN", []) or []
    isbn = meta.get("ISBN", []) or []

    # Tipos que vamos a considerar "refereed" (journal + proceedings)
    refereed_types = {"journal-article", "proceedings-article"}

    if work_type in refereed_types:
        # Confianza mayor para journal; un poco menor para proceedings por variabilidad entre conferencias
        conf = 0.9 if work_type == "journal-article" else 0.75
        return {
            "label": "refereed",
            "reason": f"crossref_type_{work_type}",
            "confidence": conf,
            "container": container,
            "publisher": publisher,
            "issn": issn,
            "isbn": isbn,
            "doi": doi,
        }

    # Otros tipos: book-chapter, posted-content, report, etc.
    return {
        "label": "unknown",
        "reason": f"crossref_type_{work_type}",
        "confidence": 0.4,
        "container": container,
        "publisher": publisher,
        "doi": doi,
    }

# Ejemplo: si ya tienes df["urls"] como lista por fila
# df["url_labels"] = df["urls"].apply(lambda lst: [classify_link_peer_review(u) for u in lst])

In [15]:
from tqdm.auto import tqdm
tqdm.pandas()

In [17]:
df["url_labels"] = df["urls_clean"].progress_apply(lambda lst: [classify_link_peer_review(u) for u in lst])

100%|██████████| 2135/2135 [03:11<00:00, 11.18it/s]


In [18]:
df.url_labels.iloc[56]

[{'label': 'unknown', 'reason': 'no_doi', 'confidence': 0.3},
 {'label': 'unknown', 'reason': 'no_doi', 'confidence': 0.3},
 {'label': 'unknown', 'reason': 'no_doi', 'confidence': 0.3},
 {'label': 'unknown', 'reason': 'no_doi', 'confidence': 0.3},
 {'label': 'not_refereed', 'reason': 'preprint_server', 'confidence': 0.95},
 {'label': 'unknown', 'reason': 'no_doi', 'confidence': 0.3}]

In [19]:
df['urls_clean'].iloc[56]

['https://www.axios.com/2025/02/11/ai-security-revamp-def-con?utm_source=openai',
 'https://www.axios.com/2025/06/26/accenture-executives-cybersecurity-ai-plans?utm_source=openai',
 'https://www.crowdstrike.com/en-us/press-releases/crowdstrike-launches-ai-red-team-services-secure-ai-systems/?utm_source=openai',
 'https://learn.microsoft.com/en-us/security/ai-red-team/training?utm_source=openai',
 'https://arxiv.org/abs/2510.02677?utm_source=openai',
 'https://www.itpro.com/technology/artificial-intelligence/openai-turns-to-red-teamers-to-prevent-malicious-chatgpt-use-as-company-warns-future-models-could-pose-high-security-risk?utm_source=openai']

In [20]:
import pandas as pd

def extract_labels_per_row(url_label_dicts) -> list[str]:
    """
    Recibe una lista de dicts (uno por URL) y devuelve una lista de labels.
    Maneja None/NaN y entradas raras.
    """
    if url_label_dicts is None or (isinstance(url_label_dicts, float) and pd.isna(url_label_dicts)):
        return []
    if not isinstance(url_label_dicts, (list, tuple)):
        return []

    labels = []
    for item in url_label_dicts:
        if isinstance(item, dict) and "label" in item:
            labels.append(item["label"])
    return labels

def label_distribution(df: pd.DataFrame, col_url_labels: str, *, by="url") -> pd.DataFrame:
    """
    Calcula la distribución de labels.

    by="url": cuenta cada URL clasificada (aplana listas) => distribución sobre todas las URLs.
    by="row": cuenta 1 label por fila (mayoría; si empate, 'mixed' o 'unknown').

    Devuelve DataFrame con count y pct.
    """
    if by not in {"url", "row"}:
        raise ValueError("by debe ser 'url' o 'row'")

    if by == "url":
        # Aplanar: cada URL aporta 1 label
        all_labels = []
        for lst in df[col_url_labels].apply(extract_labels_per_row):
            all_labels.extend(lst)

        s = pd.Series(all_labels, name="label")
        counts = s.value_counts(dropna=False)

    else:
        # Un label por fila (resumen)
        def row_label(lst):
            labels = extract_labels_per_row(lst)
            if not labels:
                return "unknown"
            vc = pd.Series(labels).value_counts()
            if len(vc) == 1:
                return vc.index[0]
            # mayoría clara
            if vc.iloc[0] > vc.iloc[1]:
                return vc.index[0]
            return "mixed"

        s = df[col_url_labels].apply(row_label)
        counts = s.value_counts(dropna=False)

    dist = counts.rename("count").to_frame()
    dist["pct"] = (dist["count"] / dist["count"].sum()).round(4)
    dist.index.name = "label"
    return dist.reset_index()

# --- Ejemplo de uso ---
# df["labels"] = df["url_labels"].apply(extract_labels_per_row)
# dist_urls = label_distribution(df, "url_labels", by="url")
# dist_rows = label_distribution(df, "url_labels", by="row")

In [21]:
df["labels"] = df["url_labels"].progress_apply(extract_labels_per_row)

100%|██████████| 2135/2135 [00:00<00:00, 367906.29it/s]


In [22]:
dist_urls = label_distribution(df, "url_labels", by="url")

In [23]:
dist_urls

,label,count,pct
0,unknown,5507,0.7568
1,not_refereed,1555,0.2137
2,refereed,215,0.0295


In [25]:
df['urls_clean'].isna().sum()

np.int64(0)

In [26]:
df

,prompt,result,references,tokens,urls_clean,url_labels,labels
0,"I want to write an article about: ""Common fair...",Several studies demonstrate the mathematical i...,['https://jmlr.org/beta/papers/v24/22-1511.htm...,NaN,[https://jmlr.org/beta/papers/v24/22-1511.html...,"[{'label': 'unknown', 'reason': 'no_doi', 'con...","[unknown, not_refereed, not_refereed, not_refe..."
1,"I want to write an article about: ""Machine Lea...",Machine learning models can exhibit bias even ...,['https://www.forbes.com/sites/aparnadhinakara...,NaN,[https://www.forbes.com/sites/aparnadhinakaran...,"[{'label': 'unknown', 'reason': 'no_doi', 'con...","[unknown, not_refereed]"
2,"I want to write an article about: ""Evaluation ...",Evaluating and mitigating fairness solely thro...,['https://arxiv.org/abs/2006.09663?utm_source=...,NaN,[https://arxiv.org/abs/2006.09663?utm_source=o...,"[{'label': 'not_refereed', 'reason': 'preprint...","[not_refereed, not_refereed, unknown, unknown,..."
3,"I want to write an article about: ""Benchmark c...",Several studies highlight how US-centric bench...,['https://davidnowak.me/why-your-ai-benchmarks...,NaN,[https://davidnowak.me/why-your-ai-benchmarks-...,"[{'label': 'unknown', 'reason': 'no_doi', 'con...","[unknown, not_refereed, not_refereed, not_refe..."
4,"I want to write an article about: ""Word embedd...",Several studies have demonstrated that word em...,['https://arxiv.org/abs/1903.03862?utm_source=...,NaN,[https://arxiv.org/abs/1903.03862?utm_source=o...,"[{'label': 'not_refereed', 'reason': 'preprint...","[not_refereed, not_refereed, not_refereed, not..."
...,...,...,...,...,...,...,...
2130,"Define ""Positive bias"". Include references.",**Positive Bias** refers to the tendency to fa...,[],NaN,[https://www.psychologytoday.com/us/basics/cog...,"[{'label': 'unknown', 'reason': 'no_doi', 'con...","[unknown, unknown, unknown]"
2131,"Define ""Governance in AI"". Include references.","""Governance in AI"" refers to the frameworks, p...",[],NaN,[https://www.oecd.org/going-digital/ai/princip...,"[{'label': 'unknown', 'reason': 'no_doi', 'con...","[unknown, unknown, unknown]"
2132,"Define ""Intrinsic hallucinations"". Include ref...",**Intrinsic hallucinations** are sensory perce...,[],NaN,[https://www.ncbi.nlm.nih.gov/pmc/articles/PMC...,"[{'label': 'unknown', 'reason': 'no_doi', 'con...","[unknown, unknown]"
2133,"Define ""Extrinsic hallucinations"". Include ref...",Extrinsic hallucinations refer to sensory perc...,[],NaN,[https://www.ncbi.nlm.nih.gov/pmc/articles/PMC...,"[{'label': 'unknown', 'reason': 'no_doi', 'con...","[unknown, unknown, unknown]"


In [27]:
df['urls_clean'].iloc[2132]

['https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6131311/',
 'https://www.frontiersin.org/articles/10.3389/fpsyt.2020.00701/full']

In [ ]:
# TODO: Revisar las etiquetas unknown y ver cómo depuramos que sea peer reviewd o no.
# MISTRAL : Mirar referencias falsas por medio de google scholar (podemos comparar con algún cálculo de diferencia
# la cita en APA con la referencia que arroja el modelo - esto para los que aparecen en scholar pero puede pasar que
# haya referencias que ni el título exista)

In [30]:
import re
import pandas as pd
import requests
from urllib.parse import urlsplit
from functools import lru_cache

DOI_RE = re.compile(r'10\.\d{4,9}/[^\s"<>]+', re.I)
URL_RE = re.compile(r'https?://[^\s<>"\']+')

def get_domain(url: str) -> str:
    try:
        return urlsplit(url).netloc.lower()
    except Exception:
        return ""

def safe_list(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    return x if isinstance(x, list) else []

def extract_urls_from_text(text) -> list[str]:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return []
    s = str(text)
    urls = []
    for m in URL_RE.finditer(s):
        u = m.group(0).strip()
        while u and u[-1] in ')]}.,;:':
            u = u[:-1]
        urls.append(u)
    # dedup preservando orden
    seen, out = set(), []
    for u in urls:
        if u and u not in seen:
            seen.add(u); out.append(u)
    return out

def extract_doi(text_or_url: str) -> str | None:
    if not text_or_url:
        return None
    m = DOI_RE.search(str(text_or_url))
    return m.group(0).rstrip(').,;:]') if m else None


# -------------------------
# Fase 1: reglas rápidas
# -------------------------

# Ajusta estas listas a tu corpus real
NOT_REFEREED_DOMAINS = {
    "axios.com", "itpro.com", "medium.com", "substack.com",
    "blog.google", "openai.com", "learn.microsoft.com",
}

# patrones de paths típicos no-académicos
NOT_REFEREED_PATH_HINTS = (
    "/blog", "/news", "/press", "/press-releases", "/newsroom",
    "/training", "/docs", "/documentation", "/help", "/support"
)

PREPRINT_DOMAINS = {"arxiv.org", "ssrn.com", "biorxiv.org", "medrxiv.org"}

def fast_rule_label(url: str) -> dict | None:
    """
    Devuelve dict con label si detecta con alta confianza.
    Si no puede concluir, devuelve None.
    """
    u = (url or "").lower()
    dom = get_domain(u)
    path = urlsplit(u).path.lower()

    if dom in PREPRINT_DOMAINS:
        return {"label": "not_refereed", "reason": "preprint_server", "confidence": 0.95}

    if dom in NOT_REFEREED_DOMAINS:
        return {"label": "not_refereed", "reason": "domain_rule", "confidence": 0.9}

    if any(h in path for h in NOT_REFEREED_PATH_HINTS):
        return {"label": "not_refereed", "reason": "path_hint_rule", "confidence": 0.8}

    return None


# -------------------------
# Fase 2: sacar DOI del HTML
# -------------------------

META_DOI_PATTERNS = [
    # citation_doi
    re.compile(r'<meta[^>]+name=["\']citation_doi["\'][^>]+content=["\']([^"\']+)["\']', re.I),
    # dc.identifier / dc.identifier.doi
    re.compile(r'<meta[^>]+name=["\']dc\.identifier(?:\.doi)?["\'][^>]+content=["\']([^"\']+)["\']', re.I),
    # og:doi (menos común)
    re.compile(r'<meta[^>]+property=["\']og:doi["\'][^>]+content=["\']([^"\']+)["\']', re.I),
]

@lru_cache(maxsize=50_000)
def fetch_html(url: str) -> str | None:
    try:
        r = requests.get(url, timeout=20, headers={"User-Agent": "unknown-refiner/1.0"})
        if r.status_code != 200:
            return None
        # Evita bajar PDFs enormes como HTML
        ctype = (r.headers.get("Content-Type") or "").lower()
        if "application/pdf" in ctype:
            return None
        return r.text
    except Exception:
        return None

def doi_from_html(url: str) -> str | None:
    html = fetch_html(url)
    if not html:
        return None

    # 1) metas
    for pat in META_DOI_PATTERNS:
        m = pat.search(html)
        if m:
            candidate = m.group(1).strip()
            d = extract_doi(candidate)
            if d:
                return d

    # 2) regex general sobre el HTML (más ruidoso pero útil)
    d = extract_doi(html)
    return d


# -------------------------
# Crossref (para DOI -> tipo)
# -------------------------

@lru_cache(maxsize=50_000)
def crossref_metadata(doi: str) -> dict | None:
    try:
        r = requests.get(
            f"https://api.crossref.org/works/{doi}",
            timeout=20,
            headers={"User-Agent": "link-classifier/1.0"}
        )
        if r.status_code != 200:
            return None
        return r.json().get("message")
    except Exception:
        return None

def classify_by_doi(doi: str) -> dict:
    meta = crossref_metadata(doi)
    if not meta:
        return {"label": "unknown", "reason": "doi_not_resolved", "confidence": 0.3, "doi": doi}

    work_type = meta.get("type")  # journal-article, proceedings-article, posted-content, etc.
    container = (meta.get("container-title") or [None])[0]
    publisher = meta.get("publisher")
    issn = meta.get("ISSN", []) or []
    isbn = meta.get("ISBN", []) or []

    if work_type == "journal-article":
        return {
            "label": "refereed",
            "reason": "crossref_journal_article",
            "confidence": 0.9,
            "doi": doi,
            "container": container,
            "publisher": publisher,
            "issn": issn,
            "isbn": isbn,
        }
    if work_type == "proceedings-article":
        return {
            "label": "refereed",
            "reason": "crossref_proceedings_article",
            "confidence": 0.75,
            "doi": doi,
            "container": container,
            "publisher": publisher,
            "issn": issn,
            "isbn": isbn,
        }

    return {
        "label": "unknown",
        "reason": f"crossref_type_{work_type}",
        "confidence": 0.4,
        "doi": doi,
        "container": container,
        "publisher": publisher,
        "issn": issn,
        "isbn": isbn,
    }


# -------------------------
# Diagnóstico de unknown
# -------------------------

def unknown_breakdown(df: pd.DataFrame, col_url_labels="url_labels") -> dict:
    """
    Devuelve:
      - dist por reason (solo unknown)
      - top dominios dentro de unknown
      - tasa unknown global (por URL)
    """
    rows = []
    for lst in df[col_url_labels].apply(safe_list):
        for item in lst:
            if isinstance(item, dict):
                url = item.get("url") or item.get("link") or ""
                label = item.get("label")
                reason = item.get("reason", "no_reason")
                rows.append({"url": url, "domain": get_domain(url), "label": label, "reason": reason})

    long = pd.DataFrame(rows)
    if long.empty:
        return {"unknown_reason_dist": pd.DataFrame(), "unknown_top_domains": pd.DataFrame(), "unknown_rate": 0.0}

    # unknown rate por URL
    unknown_rate = (long["label"].eq("unknown").mean())

    unk = long[long["label"] == "unknown"].copy()
    reason_dist = unk["reason"].value_counts().rename("count").to_frame()
    reason_dist["pct"] = (reason_dist["count"] / reason_dist["count"].sum()).round(4)
    reason_dist = reason_dist.reset_index().rename(columns={"index": "reason"})

    top_domains = unk["domain"].value_counts().head(25).rename("count").to_frame().reset_index().rename(columns={"index": "domain"})
    top_domains["pct"] = (top_domains["count"] / unk.shape[0]).round(4)

    return {
        "unknown_reason_dist": reason_dist,
        "unknown_top_domains": top_domains,
        "unknown_rate": float(round(unknown_rate, 4)),
    }


# -------------------------
# Refinamiento: actualiza unknowns
# -------------------------

def refine_unknown_item(item: dict) -> dict:
    """
    Toma un dict por URL y si está unknown, intenta reclasificar.
    Conserva la url.
    """
    if not isinstance(item, dict):
        return item

    label = item.get("label")
    url = item.get("url") or item.get("link") or item.get("source_url")
    if not url:
        return item

    # Solo trabajamos unknown
    if label != "unknown":
        # Asegura url persistente
        item["url"] = url
        return item

    # Fase 1: reglas rápidas
    fr = fast_rule_label(url)
    if fr is not None:
        out = {**item, **fr}
        out["url"] = url
        return out

    # Fase 2: DOI desde HTML (si no había DOI)
    doi = item.get("doi") or extract_doi(url)
    if not doi:
        doi = doi_from_html(url)
        if doi:
            item["doi"] = doi
            item["reason"] = "doi_found_in_html"

    if doi:
        classified = classify_by_doi(doi)
        out = {**item, **classified}
        out["url"] = url
        return out

    # Si nada funcionó, manten unknown pero actualiza razón
    out = dict(item)
    out["url"] = url
    out["reason"] = out.get("reason") or "unknown_no_signal"
    out["confidence"] = out.get("confidence", 0.3)
    return out

def refine_unknowns_df(df: pd.DataFrame, col_url_labels="url_labels", new_col=None) -> pd.DataFrame:
    """
    Refina unknowns dentro de df[col_url_labels].
    Si new_col es None, sobrescribe col_url_labels.
    """
    target_col = new_col or col_url_labels

    def refine_list(lst):
        lst = safe_list(lst)
        return [refine_unknown_item(item) for item in lst]

    df[target_col] = df[col_url_labels].apply(refine_list)
    return df


# -------------------------
# Ejemplo de uso
# -------------------------
# 1) Diagnóstico inicial
# stats = unknown_breakdown(df, col_url_labels="url_labels")
# stats["unknown_rate"], stats["unknown_reason_dist"].head(), stats["unknown_top_domains"].head()

# 2) Refinar unknowns
# df = refine_unknowns_df(df, col_url_labels="url_labels")  # sobrescribe
# stats2 = unknown_breakdown(df, col_url_labels="url_labels")

In [31]:
stats = unknown_breakdown(df, col_url_labels="url_labels")

In [32]:
stats

{'unknown_reason_dist':                         reason  count     pct
 0                       no_doi   4908  0.8912
 1             doi_not_resolved    586  0.1064
 2         crossref_type_report      8  0.0015
 3    crossref_type_proceedings      3  0.0005
 4  crossref_type_journal-issue      1  0.0002
 5           crossref_type_book      1  0.0002,
 'unknown_top_domains':   domain  count  pct
 0          5507  1.0,
 'unknown_rate': 0.7568}

In [33]:
df = refine_unknowns_df(df, col_url_labels="url_labels")

In [35]:
stats2 = unknown_breakdown(df, col_url_labels="url_labels")

In [36]:
stats2

{'unknown_reason_dist':                         reason  count     pct
 0                       no_doi   4908  0.8912
 1             doi_not_resolved    586  0.1064
 2         crossref_type_report      8  0.0015
 3    crossref_type_proceedings      3  0.0005
 4  crossref_type_journal-issue      1  0.0002
 5           crossref_type_book      1  0.0002,
 'unknown_top_domains':   domain  count  pct
 0          5507  1.0,
 'unknown_rate': 0.7568}

In [ ]:
# TODO: para los que no están en peer reviewd, revisar si existe su versión publicada
